In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import expm,svd
import math
import time
from os.path import exists


def vec_conj(a):
    return np.conjugate(a)

def vec_dot(a,b):
    # second vector is conjuagated
    return np.dot(a,vec_conj(b))

def dagger(M):
    return np.conjugate(np.transpose(M))

def mat_dot2(A,B):
    return np.dot(A,B)

def mat_dot4(A,B,C,D):
    return np.dot(A,np.dot(B,np.dot(C,D)))

def normalize(psi):
    return psi/np.sqrt(vec_dot(psi,psi))

def I_3():
    I3 = np.eye(3, dtype=complex)
    return I3

def l_1():
    l1 = np.matrix([[0,1.,0],[1.,0,0],[0,0,0]])
    return l1

def l_2():
    l2 = np.matrix([[0,-1j,0],[1j,0,0],[0,0,0]])
    return l2

def l_3():
    l3 = np.matrix([[0,0,0],[0,1.,0],[0,0,1.]])
    return l3

def l_4():
    l4 = np.matrix([[0,0,1.],[0,0,0],[1.,0,0]])
    return l4

def l_5():
    l5 = np.matrix([[0,0,-1j],[0,0,0],[1j,0,0]])
    return l5

def l_6():
    l6 = np.matrix([[0,0,0],[0,0,1.],[0,1.,0]])
    return l6

def l_7():
    l7 = np.matrix([[0,0,0],[0,0,-1j],[0,1j,0]])
    return l7

def l_8():
    l8 = np.matrix([[0,0,0],[0,1.,0],[0,0,-1.]])
    return l8

def gellmann_normal():
    return [np.eye(3),np.array(l_1()),np.array(l_2()),np.array(l_3()),np.array(l_4()),np.array(l_5()),np.array(l_6()),np.array(l_7()),np.array(l_8())]

def gellmann_tilde():
    return [np.eye(3),g*np.array(l_1()),g*np.array(l_2()),g*np.array(l_3()),g*np.array(l_4()),g*np.array(l_5()),g*np.array(l_6()),g*np.array(l_7()),g*np.array(l_8())]

def gellmann_bar():
    return [np.eye(3),(1/g)*np.array(l_1()),(1/g)*np.array(l_2()),(1/g)*np.array(l_3()),(1/g)*np.array(l_4()),(1/g)*np.array(l_5()),(1/g)*np.array(l_6()),(1/g)*np.array(l_7()),(1/g)*np.array(l_8())]


# def Hamiltonian
# local spin ladder operators
S_a_plus  = (l_1() - 1j * l_2()) / 2
S_a_minus = (l_1() + 1j * l_2()) / 2
S_b_plus  = (l_4() - 1j * l_5()) / 2
S_b_minus = (l_4() + 1j * l_5()) / 2

# local number and magnetization operators
n_a = (l_3() + l_8()) / 2
n_b = (l_3() - l_8()) / 2
n_loc = l_3()
m_loc = l_8()

def draw_dis(L, mean, W, rng = None):
    rng = np.random.default_rng() if rng is None else rng
    return rng.normal(loc=mean, scale=W, size=L)


def draw_q(L, mean, W, rng=None):
    rng=np.random.default_rng() if rng is None else rng
    return rng.normal(loc=mean, scale=W, size=(L, 2))


def build_MPO(L, t, Omega_list, q_list, V_list):
    """
    V_list[j] is the coupling on the bond (j, j+1), so effectively length L-1.
    Omega_list and q_list have length L and shape (L, 2) respectively.
    """
    d = 3
    w = 7

    def make_W(Omega_j, q_a, q_b, V_j):
        W = np.zeros((w, w, d, d), dtype=complex)

        # --- Right column (finishing operator B_{j+1} of each two-site term) ---
        W[0, 0] = I_3           # identity channel (term already completed to the right)
        W[1, 0] = S_a_plus     # finishes  y-_j y+_{j+1}   (hopping, flavor a)
        W[2, 0] = S_a_minus    # finishes  y+_j y-_{j+1}   (hopping, flavor a)
        W[3, 0] = S_b_plus     # finishes  y-_j y+_{j+1}   (hopping, flavor b)
        W[4, 0] = S_b_minus    # finishes  y+_j y-_{j+1}   (hopping, flavor b)
        W[5, 0] = m_loc        # finishes  m_j m_{j+1}     (V interaction)

        # --- Bottom row (starting operator A_j, carries the coefficient) ---
        W[6, 1] = -t * S_a_minus
        W[6, 2] = -t * S_a_plus
        W[6, 3] = -t * S_b_minus
        W[6, 4] = -t * S_b_plus
        W[6, 5] = V_j * m_loc
        W[6, 6] = I_3           # identity channel (no term started yet from the left)

        # --- Bottom-right corner: purely on-site terms, born and die at site j ---
        W[6, 0] = (Omega_j * (S_a_plus @ S_b_minus + S_b_plus @ S_a_minus)
                   + q_a * n_a + q_b * n_b)
        return W

    v_L = np.zeros(w, dtype=complex); v_L[6] = 1.0
    v_R = np.zeros(w, dtype=complex); v_R[0] = 1.0

    # V_j lives on bond (j, j+1); site L-1 has no bond to its right, so V is 0 there.
    Ws = [make_W(Omega_list[j], q_list[j, 0], q_list[j, 1],
                 V_list[j] if j < L - 1 else 0.0) for j in range(L)]

    MPO = [None] * L
    MPO[0]  = np.einsum("a,abij->bij", v_L, Ws[0])
    MPO[-1] = np.einsum("abij,b->aij", Ws[-1], v_R)
    for j in range(1, L - 1):
        MPO[j] = Ws[j]

    return MPO

def build_onsite_omega(Omega_j, q_a, q_b):
    """3x3 on-site operator: omega_j = Omega_j (a+ b- + b+ a-) + q_a n_a + q_b n_b."""
    return (Omega_j * (S_a_plus @ S_b_minus + S_b_plus @ S_a_minus)
            + q_a * n_a + q_b * n_b)


def build_bond_hamiltonian_twosite(t, V_i):
    """9x9 bare two-site piece on bond (i, i+1): hopping (both flavors) and V_i m m."""
    h = np.zeros((9, 9), dtype=complex)
    h += -t * (np.kron(S_a_plus, S_a_minus) + np.kron(S_a_minus, S_a_plus))
    h += -t * (np.kron(S_b_plus, S_b_minus) + np.kron(S_b_minus, S_b_plus))
    h +=  V_i * np.kron(m_loc, m_loc)
    return h


def build_bond_hamiltonians_tilde(L, t, Omega_list, q_list, V_list):
    if L < 2:
        raise ValueError("Need at least L=2 to define bond Hamiltonians.")

    omegas = [build_onsite_omega(Omega_list[j], q_list[j, 0], q_list[j, 1])
              for j in range(L)]

    h_tilde_list = []
    for i in range(L - 1):
        # Weights on the two sites of this bond, aware of open-boundary edges.
        w_left  = 1.0 if i == 0         else 0.5
        w_right = 1.0 if (i + 1) == L-1 else 0.5
        h = build_bond_hamiltonian_twosite(t, V_list[i])
        h += w_left  * np.kron(omegas[i],     I_3)
        h += w_right * np.kron(I_3, omegas[i + 1])
        h_tilde_list.append(h)
    return h_tilde_list


def _expm_hermitian(H, theta):
    # Symmetrize to kill roundoff drift before eigh.
    Hs = 0.5 * (H + H.conj().T)
    w, U = np.linalg.eigh(Hs)
    return (U * np.exp(-1j * theta * w)) @ U.conj().T


def build_bond_gates(L, t, Omega_list, q_list, V_list, tau):
    h_tilde_list = build_bond_hamiltonians_tilde(L, t, Omega_list, q_list, V_list)

    gates_odd  = []   # bonds (0,1), (2,3), (4,5), ...  ← bond index 2k
    gates_even = []   # bonds (1,2), (3,4), (5,6), ...  ← bond index 2k+1
    for i, h in enumerate(h_tilde_list):
        if i % 2 == 0:
            gates_odd.append(_expm_hermitian(h, tau / 2.0))
        else:
            gates_even.append(_expm_hermitian(h, tau))
    return gates_odd, gates_even

# build random initial state
def random_config(L, N, Na, rng=None):
    """
    Returns a random configuration of length L with exactly N particles,
    Na of which are flavor a (|a>=1) and N-Na are flavor b (|b>=2).
    The remaining L-N sites are |vac> (0).
    """
    if rng is None:
        rng = np.random.default_rng()
    config = np.zeros(L, dtype=int)
    particle_sites = rng.choice(L, size=N, replace=False)
    flavors = np.array([1] * Na + [2] * (N - Na))
    rng.shuffle(flavors)
    config[particle_sites] = flavors
    return config.tolist()


def build_product_state(L, psi0_config, chi_init=1):
    """
    psi0_config[j] in {0, 1, 2}, meaning the local basis index at site j:
      0 -> |vac>, 1 -> |a>, 2 -> |b>.
    """
    d = 3
    if len(psi0_config) != L:
        raise ValueError(f"psi0_config length {len(psi0_config)} != L={L}")
    
    MPS = [None] * L
    for i, occ in enumerate(psi0_config):
        if occ not in (0, 1, 2):
            raise ValueError(f"site {i}: occ must be 0, 1, or 2 (got {occ!r})")
        A = np.zeros((chi_init, d, chi_init), dtype=complex)
        A[0, occ, 0] = 1.0
        MPS[i] = A
    
    vL = np.zeros(chi_init); vL[0] = 1.0
    vR = np.zeros(chi_init); vR[0] = 1.0
    MPS[0]  = np.einsum('a, asb -> sb', vL, MPS[0])
    MPS[-1] = np.einsum('asb, b -> as', MPS[-1], vR)
    return MPS

def U_mat(dt,U):
#     Hi = transverse_ising(J,hx,hz)
#     U = expm(-1j*dt*Hi)
    Ud = dagger(U)
    U_all = np.zeros((9,9,9,9),dtype=np.complex128)
    for i in range(9):
        for j in range(9):
            for k in range(9):
                for l in range(9):
                    sg1 = np.kron(gellmann_bar()[i],gellmann_bar()[j])
                    sg2 = np.kron(gellmann_tilde()[k],gellmann_tilde()[l])
                    U_all[i][j][k][l] = (1/9)*np.trace(mat_dot4(sg1,U,sg2,Ud))
    return U_all


def initial_MPDO_dict(L,seed,): # N, Na
    A_dict = {}
    # disorder
    for i in range(L):
        A_temp = np.zeros(9,dtype=np.complex128)
        psi = # local wavefunction at site i 
        psi = normalize(psi)
        rho = np.outer(psi,np.conjugate(psi))
        #print(np.trace(rho))
        for j in range(9):
            A_temp[j] = np.trace(mat_dot2(gellmann_bar()[j],rho))
        key = str("A"+str(i))
        A_dict[key] = np.reshape(A_temp,(1,9,1))
    return A_dict


class MPDO:
    # Hamiltonian parameters
    L = 10
    N = 7
    Na = 3
    t_hop = 1.0
    W = 0.2
    dt = 0.01
    T = 0.25
    cutoff = 1e-16
    seed = 42
    d = 3
    chi = 32
    
    def __init__(self,L,N,Na,t_hop,W,dt,T,chi,seed):
        self.L = L
        self.N = N
        self.Na = Na
        self.t_hop = t_hop
        self.W = W
        self.chi = chi
        self.T = T
        self.N = N
        # self.dt = self.T/self.N
        self.dt = dt        
        self.seed = seed

        self.Omega_list = draw_dis(self.L, 0, self.W, self.seed)
        self.V_list = draw_dis(self.L - 1, 0, self.W, self.seed)
        self.q_list = draw_q(self.L, 0, self.W)

        # Initialize U
        self.odd_bonds_U, self.even_bonds_U = build_bond_gates(self.L,
                                                               self.t_hop,
                                                               self.Omega_list,
                                                               self.q_list,
                                                               self.V_list,
                                                               self.dt)
        # Initial state
        self.config = random_config(self.L, self.N, self.Na, self.seed) # if we want to run different random disorder in H 
                                                                        # but keep the same initial state, change self.seed
                                                                        # to a fixed seed
        self.psi0   = build_product_state(self.L, self.config)
        


        if bound_diff == False:
            Hi = transverse_ising(self.J,self.hx,self.hz)
            self.sU = expm(-1j*self.dt*Hi)
            self.U = U_mat(self.dt,self.sU)
        elif bound_diff == True:
            Hi,Hi_start,Hi_end = transverse_ising(self.J,self.hx,self.hz)
            self.sU = expm(-1j*self.dt*Hi)
            self.U = U_mat(self.dt,self.sU)
            self.sU_start = expm(-1j*self.dt*Hi_start)
            self.U_start = U_mat(self.dt,self.sU_start)
            self.sU_end = expm(-1j*self.dt*Hi_end)
            self.U_end = U_mat(self.dt,self.sU_end)
#         self.U = U_mat(self.L,self.J,self.hx,self.hz,self.dt)
#         self.sU = U_mat(self.L,self.J,self.hx,self.hz,self.dt)[0]
        
        # Initialize MPS
        self.A_dict = initial_MPDO_dict(self.L)
        self.lmbd_position = 0
        
        # Initialize Schrodinger wavefunction - all spins up
        # if sch_bool == True:
        #     self.schrodinger_psi = initial_psi_sch(self.L)
        #     self.msr_schrodinger_mui = np.zeros((3,self.L),dtype=np.complex128)
        #     self.E_persite_sch = np.zeros(self.L-1,dtype=np.complex128)
        #     self.E_total_sch = 0
        #     self.renyi_test_left = 0
        #     self.renyi_test_right = 0
        #     self.E_fourier_sch = 0
        

        # initialize all your measurements
        self.tr_TEBD = 0
        self.norm_trace = 0
        self.renyi_left = 0
        self.renyi_right = 0
        self.renyi_full = 0
        self.msr_mui = np.zeros((3,self.L),dtype=np.complex128)
        self.E_persite = np.zeros(self.L-1,dtype=np.complex128)
        self.E_total_TEBD = 0
        self.sq_trace = 0
        self.E_fourier = 0
        
    # Updates the Schrodinger wavefunction
    def applyU_schrodinger(self, ind, sU): # U is 4x4
        self.schrodinger_psi = np.reshape(self.schrodinger_psi, (2**ind,4,2**(self.L-2-ind)))
        self.schrodinger_psi = np.einsum('ij, ajb -> aib', sU, self.schrodinger_psi,optimize='optimal')
        self.schrodinger_psi = self.schrodinger_psi.flatten()
        
    def applyU(self,ind,dirc,U,sU,lm=False):
        
        # This part relocates lmbd to the right position
        if dirc == 'left':
            self.lmbd_relocate(ind[1])
        elif dirc == 'right':
            self.lmbd_relocate(ind[0])
        
        # Need to evolve the schrodinger picture when lm = False
        if lm == False and sch_bool == True:
            self.applyU_schrodinger(ind[0], sU)
            
        A1 = self.A_dict["A"+str(ind[0])]
        A2 = self.A_dict["A"+str(ind[1])]
        chi1 = np.shape(A1)[0]
        chi2 = np.shape(A2)[2]
        
        s1 = np.einsum('ijkl,akb,blc->aijc',U,A1,A2,optimize='optimal')
        
        s2 = np.reshape(s1,(9*chi1,9*chi2))
        try:
            Lp,lmbd,R=np.linalg.svd(s2,full_matrices=False)
        except np.linalg.LinAlgError as err:
            if "SVD did not converge" in str(err):
                Lp,lmbd,R=svd(s2,full_matrices=False,lapack_driver='gesvd')
                f = open("py_print.txt","a")
                f.write("SVD convergence issue")
                f.close()
            else:
                raise
        chi12 = np.min([9*chi1,9*chi2])
        chi12_p = np.min([self.chi,chi12])
        lmbd = np.diag(lmbd)
    
        # Truncation step
        lmbd = lmbd[:chi12_p,:chi12_p]
        Lp = Lp[:,:chi12_p]
        R = R[:chi12_p,:]
    
        if (dirc == 'left'):
            A1 = np.reshape(np.dot(Lp,lmbd),(chi1,4,chi12_p))
            A2 = np.reshape(R,(chi12_p,4,chi2))
            self.lmbd_position = ind[0]
            
        elif (dirc == 'right'):
            A1 = np.reshape(Lp,(chi1,4,chi12_p))
            A2 = np.reshape(np.dot(lmbd,R),(chi12_p,4,chi2))
            self.lmbd_position = ind[1]
            
        self.A_dict["A"+str(ind[0])] = A1
        self.A_dict["A"+str(ind[1])] = A2
        
    # Function to move lmbd right
    def move_lmbd_right(self,ind):
        I = np.reshape(np.eye(16),(4,4,4,4))
        self.applyU([ind,ind+1],'right',I,0,lm=True)
    
    # Function to move lmbd left
    def move_lmbd_left(self,ind):
        I = np.reshape(np.eye(16),(4,4,4,4))
        self.applyU([ind,ind+1],'left',I,0,lm=True)
        
    def sweepU(self):
    
        even_sites = [[i,i+1] for i in np.arange(0,self.L-1,2)]
        odd_sites = [[i,i+1] for i in np.arange(1,self.L-1,2)]
        odd_sites.reverse()
        
        for i in even_sites:
            if bound_diff == False:
                self.applyU(i,'right',self.U,self.sU)
            else:
                if i == [0,1]:
                    self.applyU(i,'right',self.U_start,self.sU_start)
                elif i == [self.L-2,self.L-1]:
                    self.applyU(i,'right',self.U_end,self.sU_end)
                else:
                    self.applyU(i,'right',self.U,self.sU)
            
        for i in odd_sites:
            self.applyU(i,'left',self.U,self.sU)
        
        # if sch_bool == True:
        #     self.measure_schrodinger()
        self.measure_TEBD()
    
    # Relocates lmbd from lmbd_position to ind
    def lmbd_relocate(self,ind):
        step = ind - self.lmbd_position
        for i in range(np.abs(step)):
            if step > 0:
                self.move_lmbd_right(self.lmbd_position)
            elif step < 0:
                self.move_lmbd_left(self.lmbd_position-1)
                
    # Measures the expectation value using the Schrodinger wavefunction
    def measure_schrodinger(self):
        
        # Spin
        for m in range(1,4):
            for ind in range(self.L):
                #msr_sch = np.array((1/2)*pauli_matrices(self.L,ind)[m]) # fixx this
                msr_psi = np.reshape(self.schrodinger_psi,(2**ind,2,2**(self.L-1-ind)))
                #self.msr_schrodinger_mui[m-1][ind] = np.dot(np.conjugate(self.schrodinger_psi),np.dot(msr_sch,self.schrodinger_psi))
                self.msr_schrodinger_mui[m-1][ind] = np.einsum('aib,ij,ajb',np.conjugate(msr_psi),spin_all()[m],msr_psi)
        
        # Energy
#         self.E_total_sch = 0
#         for i in range(self.L-1):
#             msr_e = np.array(transverse_ising_psite(self.L,self.J,self.hx,self.hz)[i])
#             self.E_persite_sch[i] = np.dot(np.conjugate(self.schrodinger_psi),np.dot(msr_e,self.schrodinger_psi))
#             self.E_total_sch += self.E_persite_sch[i]
            
        # Different way of measuring the energy
        self.E_total_sch = 0
        Sz_ij = np.zeros(self.L-1,dtype=np.complex128)
        for ind in range(self.L-1):
            trunc_psi = np.reshape(self.schrodinger_psi,(2**ind,4,2**(self.L-2-ind)))
            Sz_ij[ind] = np.einsum('aib,ij,ajb',np.conjugate(trunc_psi),np.kron(spin_all()[3],spin_all()[3]),trunc_psi)
            #Sz_ij[ind] = np.dot(np.conjugate(self.schrodinger_psi),np.reshape(np.dot(pauli_SzSz(self.L,ind),self.schrodinger_psi),(2**L,1)))
            #print(Sz_ij)
        self.E_persite_sch[0] = (self.hx/2)*self.msr_schrodinger_mui[0][0] + (self.hx/4)*self.msr_schrodinger_mui[0][1] + self.J*Sz_ij[0]+(self.hz/2)*self.msr_schrodinger_mui[2][0] + (self.hz/4)*self.msr_schrodinger_mui[2][1]
        self.E_persite_sch[-1] = (self.hx/2)*self.msr_schrodinger_mui[0][-1] + (self.hx/4)*self.msr_schrodinger_mui[0][-2] + self.J*Sz_ij[-1] +(self.hz/2)*self.msr_schrodinger_mui[2][-1] + (self.hz/4)*self.msr_schrodinger_mui[2][-2]
        for i in range(1,self.L-2):
            self.E_persite_sch[i] = (self.hx/4)*self.msr_schrodinger_mui[0][i] + (self.hx/4)*self.msr_schrodinger_mui[0][i+1] + self.J*Sz_ij[i] +(self.hz/4)*self.msr_schrodinger_mui[2][i] + (self.hz/4)*self.msr_schrodinger_mui[2][i+1]
        for i in range(0,self.L-1):
            self.E_total_sch += self.E_persite_sch[i]
        
        self.E_fourier_sch = 0
        for i in range(self.L-1):
            self.E_fourier_sch += np.e**(1j*(i+1)*self.k)*self.E_persite_sch[i]
        self.E_fourier_sch = -(1/self.L)*self.E_fourier_sch
        
        
        # Total energy
#         H_tot = np.zeros((2**L,2**L),dtype=np.complex128)
#         for i in range(self.L-1):
#             H_tot += transverse_ising_psite(self.L,self.J,self.hx,self.hz)[i]
#         self.E_total_sch = np.dot(np.conjugate(self.schrodinger_psi),np.dot(H_tot,self.schrodinger_psi))
        
        # Renyi test
        psi_mat = np.reshape(self.schrodinger_psi,(int(2**(self.L/2)),int(2**(self.L/2))))
        rho_left = np.dot(dagger(psi_mat),psi_mat)
        rho_right = np.dot(psi_mat,dagger(psi_mat))
        tr_left_test = np.trace(np.dot(rho_left,rho_left))
        tr_right_test = np.trace(np.dot(rho_right,rho_right))
        total_trace = np.trace(rho_left)
        self.renyi_test_left = -np.log2(tr_left_test/(total_trace**2))
        self.renyi_test_right = -np.log2(tr_right_test/(total_trace**2))
                
    def measure_TEBD(self):
        
        self.left_trace = []
        self.right_trace = []
        self.build_left()
        self.build_right()
        
        for m in range(1,4):
            
            for ind in range(self.L):
                temp = self.left_trace[ind]
                temp = np.tensordot(temp,g*self.A_dict["A"+str(ind)][:,m,:],axes=1)
                temp = np.tensordot(temp,self.right_trace[ind],axes=1)
                self.msr_mui[m-1][ind] = (1/2)*temp.flatten()[0]
        
        # Measure trace
        temp = self.A_dict["A0"][:,0,:]
        for i in range(1,self.L):
            temp = np.tensordot(temp,self.A_dict["A"+str(i)][:,0,:],axes=1)
        temp = temp.flatten()[0]
        self.tr_TEBD = temp
    
                
        # Measure energy per site
        # Measure SzSz
        Sz_ij = np.zeros(self.L-1,dtype=np.complex128)
        for ind in range(self.L-1):
            Sz_ij[ind] = self.tensordot_SzSz(ind)
    
        self.E_total_TEBD = 0
        for i in range(0,self.L-1):
            if i == 0:
                self.E_persite[i] = (self.hx/2)*self.msr_mui[0][i] + (self.hx/4)*self.msr_mui[0][i+1] + self.J*Sz_ij[i] +(self.hz/2)*self.msr_mui[2][i] + (self.hz/4)*self.msr_mui[2][i+1]
            elif i == L-2:
                self.E_persite[i] = (self.hx/4)*self.msr_mui[0][i] + (self.hx/2)*self.msr_mui[0][i+1] + self.J*Sz_ij[i] +(self.hz/4)*self.msr_mui[2][i] + (self.hz/2)*self.msr_mui[2][i+1]
            else:
                self.E_persite[i] = (self.hx/4)*self.msr_mui[0][i] + (self.hx/4)*self.msr_mui[0][i+1] + self.J*Sz_ij[i] +(self.hz/4)*self.msr_mui[2][i] + (self.hz/4)*self.msr_mui[2][i+1]
            self.E_total_TEBD += self.E_persite[i]
            
        self.E_fourier = 0
        for i in range(self.L-1):
            self.E_fourier += np.e**(1j*(i+1)*self.k)*self.E_persite[i]
        self.E_fourier = -(1/self.L)*self.E_fourier
    
    def build_left(self):
        temp = np.reshape(1.+0.*1j,(1,1))
        self.left_trace.append(temp)
        for i in range(1,self.L):
            temp = np.tensordot(temp,self.A_dict["A"+str(i-1)][:,0,:],axes=1)
            self.left_trace.append(temp)
        
    def build_right(self):
        temp = np.reshape(1.+0.*1j,(1,1))
        self.right_trace.append(temp)
        loop_arr = np.arange(self.L-2,-1,-1)
        for i in loop_arr:
            temp = np.tensordot(self.A_dict["A"+str(i+1)][:,0,:],temp,axes=1)
            self.right_trace.append(temp)
        self.right_trace.reverse()
        
    def tensordot_A(self, ind):
        # ind is an array
        temp = self.A_dict["A0"][:,int(ind[0]),:]
        for i in range(1,self.L):
            temp = np.tensordot(temp,self.A_dict["A"+str(i)][:,int(ind[i]),:],axes=1)
            
        return temp.flatten()[0]
    
    def tensordot_SzSz(self, ind):
        # Sz at ind and ind+1
        temp = self.left_trace[ind]
        temp = np.tensordot(temp,(1/2)*g*self.A_dict["A"+str(ind)][:,3,:],axes=1)
        temp = np.tensordot(temp,(1/2)*g*self.A_dict["A"+str(ind+1)][:,3,:],axes=1)
        temp = np.tensordot(temp,self.right_trace[ind+1],axes=1)
        return temp.flatten()[0]
    


g = #GG#
chi = #CHI#
bound_diff = True
sch_bool = False
_renyi = False
_sqtrace = False
L = #LL#
T = #TT#
N = #NN#

Sz_TEBD = np.zeros((L,N),dtype=np.complex128)
Sy_TEBD = np.zeros((L,N),dtype=np.complex128)
Sx_TEBD = np.zeros((L,N),dtype=np.complex128)
E_psite = np.zeros((L-1,N),dtype=np.complex128)
Sz_schrodinger = []
tr_TB = []
tr2_TB = []
Et_TEBD = []
Ef_TEBD = []
if _sqtrace == True:
    norm_trace_TEBD = []
    renyi_F = []
if _renyi == True:
    renyi_L = []
    renyi_R = []

if sch_bool == True:
    Sz_sch = np.zeros((L,N),dtype=np.complex128)
    Sy_sch = np.zeros((L,N),dtype=np.complex128)
    Sx_sch = np.zeros((L,N),dtype=np.complex128)
    E_psite_sch = np.zeros((L-1,N),dtype=np.complex128)
    Et_sch = []
    renyi_L_test = []
    renyi_R_test = []
    Ef_sch = []

mps_evolve = MPDO(L,chi,T,N)
if exists("py_print.txt"):
    f = open("py_print.txt","w")
    f.write('New run\n')
    f.close
else:
    f = open("py_print.txt","x")
    f.write('New run\n')
    f.close()
for i in range(N):

    t1 = time.time()
    mps_evolve.sweepU()
    t2 = time.time()
    f = open("py_print.txt","a")
    f.write('Time step = '+str(i)+', time taken = '+str(t2-t1)+' for each step\n')
    #print('Time step = '+str(i)+', time taken = '+str(t2-t1)+' for each step')
    f.close()
    for j in range(L):
        Sx_TEBD[j][i] = mps_evolve.msr_mui[0][j]
        Sy_TEBD[j][i] = mps_evolve.msr_mui[1][j]
        Sz_TEBD[j][i] = mps_evolve.msr_mui[2][j]
        if j != L-1:
            E_psite[j][i] = mps_evolve.E_persite[j]
    Et_TEBD.append(mps_evolve.E_total_TEBD)
    tr_TB.append(mps_evolve.tr_TEBD)
    Ef_TEBD.append(mps_evolve.E_fourier)
    if _sqtrace == True:
        norm_trace_TEBD.append(mps_evolve.norm_trace)
        tr2_TB.append(mps_evolve.sq_trace)
        renyi_F.append(mps_evolve.renyi_full)
    if _renyi == True:
        renyi_L.append(mps_evolve.renyi_left)
        renyi_R.append(mps_evolve.renyi_right)
    
    if sch_bool == True:
        for j in range(L):
            Sx_sch[j][i] = mps_evolve.msr_schrodinger_mui[0][j]
            Sy_sch[j][i] = mps_evolve.msr_schrodinger_mui[1][j]
            Sz_sch[j][i] = mps_evolve.msr_schrodinger_mui[2][j]
            if j != L-1:
                E_psite_sch[j][i] = mps_evolve.E_persite_sch[j]
        Et_sch.append(mps_evolve.E_total_sch)
        Ef_sch.append(mps_evolve.E_fourier_sch)
        renyi_L_test.append(mps_evolve.renyi_test_left)
        renyi_R_test.append(mps_evolve.renyi_test_right)
    
# Saving data
if sch_bool == True:
    final_array = np.zeros((N,8*L+10))
else:
    final_array = np.zeros((N,4*L+2))
col_names = []
for i in range(L):
    col_names.append('Sx'+str(i))
for i in range(L):
    col_names.append('Sy'+str(i))
for i in range(L):
    col_names.append('Sz'+str(i))
for i in range(L-1):
    col_names.append('E'+str(i))
col_names.append('E-total')
col_names.append('Trace')
#col_names.append('Trace^2')
#col_names.append('Norm_trace')
#col_names.append('RenyiF')
#col_names.append('RenyiL')
#col_names.append('RenyiR')
col_names.append('E_fourier')

if sch_bool == True:
    for i in range(L):
        col_names.append('Sx_sch'+str(i))
    for i in range(L):
        col_names.append('Sy_sch'+str(i))
    for i in range(L):
        col_names.append('Sz_sch'+str(i))
    for i in range(L-1):
        col_names.append('E_sch'+str(i))
    col_names.append('E_sch-total')
    col_names.append('Renyi_schL')
    col_names.append('Renyi_schR')
    col_names.append('E_fourier_sch')

df = pd.DataFrame(final_array,columns=np.array(col_names))

for i in range(L):
    temp_list_Sx = Sx_TEBD[i]
    df['Sx'+str(i)][:] = np.real(temp_list_Sx)

for i in range(L):
    temp_list_Sy = Sy_TEBD[i]
    df['Sy'+str(i)][:] = np.real(temp_list_Sy)

for i in range(L):
    temp_list_Sz = Sz_TEBD[i]
    df['Sz'+str(i)][:] = np.real(temp_list_Sz)
    
for i in range(L-1):
    temp_list_E = E_psite[i]
    df['E'+str(i)][:] = np.real(temp_list_E)
    
temp_list_Et = np.array(Et_TEBD)
df['E-total'][:] = np.real(temp_list_Et)

temp_list_trace = np.array(tr_TB)
df['Trace'][:] = np.real(temp_list_trace)

#temp_list_tr2 = np.array(tr2_TB)
#df['Trace^2'][:] = np.real(temp_list_tr2)

#temp_list_normt = np.array(norm_trace_TEBD)
#df['Norm_trace'][:] = np.real(temp_list_normt)

#temp_list_renyiF = np.array(renyi_F)
#df['RenyiF'][:] = np.real(temp_list_renyiF)

#temp_list_renyiL = np.array(renyi_L)
#df['RenyiL'][:] = np.real(temp_list_renyiL)

#temp_list_renyiF = np.array(renyi_R)
#df['RenyiR'][:] = np.real(temp_list_renyiF)

df['E_fourier'][:] = np.real(np.array(Ef_TEBD))

if sch_bool == True:
    for i in range(L):
        df['Sx_sch'+str(i)][:] = np.real(Sx_sch[i])
    for i in range(L):
        df['Sy_sch'+str(i)][:] = np.real(Sy_sch[i])
    for i in range(L):
        df['Sz_sch'+str(i)][:] = np.real(Sz_sch[i])
    for i in range(L-1):
        df['E_sch'+str(i)][:] = np.real(E_psite_sch[i])
    df['E_sch-total'][:] = np.real(np.array(Et_sch))
    df['Renyi_schL'][:] = np.real(np.array(renyi_L_test))
    df['Renyi_schR'][:] = np.real(np.array(renyi_R_test))
    df['E_fourier_sch'][:] = np.real(np.array(Ef_sch))
    
df.to_csv('MPDO_DMT_lowe_L'+str(L)+'_chi'+str(chi)+'_g'+str(g)+'_T'+str(T)+'_N'+str(N)+'.csv')


